In [1]:
import pandas as pd

# Dateipfad
file_path = r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_1991-2025.xlsx"

# Excel-Datei einlesen
df = pd.read_excel(file_path)

# Erste Zeilen anzeigen
print(df.head())

# Informationen zum DataFrame anzeigen
print(df.info())

  Wirtschaftsjahr Bilanz Produkt Bilanz Oberbegriff  \
0         1991/92         Gerste     Detailposition   
1         1991/92         Gerste     Detailposition   
2         1991/92         Gerste     Detailposition   
3         1991/92         Gerste     Detailposition   
4         1991/92         Gerste     Detailposition   

                       Bilanz Position          Bilanz Einheit  Bilanz Wert  
0          Inlandsverwendung insgesamt  1 000 t Produktgewicht      10637.0  
1    Nahrungsverbrauch in Getreidewert    1 000 t Getreidewert         10.0  
2  Industrielle Verwertung für Energie  1 000 t Produktgewicht          NaN  
3                              Saatgut  1 000 t Produktgewicht        388.0  
4              Industrielle Verwertung  1 000 t Produktgewicht       2671.0  
<class 'pandas.DataFrame'>
RangeIndex: 1632 entries, 0 to 1631
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Wir

In [4]:
print(df.columns.tolist())

display(df[["Bilanz Produkt", "Bilanz Position"]].head(10))

['Wirtschaftsjahr', 'Bilanz Produkt', 'Bilanz Oberbegriff', 'Bilanz Position', 'Bilanz Einheit', 'Bilanz Wert']


,Bilanz Produkt,Bilanz Position
0,Gerste,Inlandsverwendung insgesamt
1,Gerste,Nahrungsverbrauch in Getreidewert
2,Gerste,Industrielle Verwertung für Energie
3,Gerste,Saatgut
4,Gerste,Industrielle Verwertung
5,Gerste,Verluste
6,Gerste,Nahrungsverbrauch Ausbeute in %
7,Gerste,Futter
8,Gerste,Nahrungsverbrauch in Getreidewert in kg je Kopf
9,Gerste,Selbstversorgungsgrad in %


In [5]:
import pandas as pd

# 1. Identify the two relevant balance positions
position_total = "Inlandsverwendung insgesamt"
position_alt = "Inlandsverwendung"
position_new = "Inlandsverwendung standardisiert"

# 2. Filter the relevant rows
domestic_total = df[df["Bilanz Position"] == position_total].copy()
domestic_alt = df[df["Bilanz Position"] == position_alt].copy()

# 3. Use total domestic use as the primary source
#    and fill missing values with the alternative position
keys = ["Wirtschaftsjahr", "Bilanz Produkt"]

total_values = domestic_total[keys + ["Bilanz Wert"]].rename(
    columns={"Bilanz Wert": "Wert insgesamt"}
)

alt_values = domestic_alt[keys + ["Bilanz Wert"]].rename(
    columns={"Bilanz Wert": "Wert alternativ"}
)

standardized = total_values.merge(
    alt_values,
    on=keys,
    how="outer"
)

standardized["Bilanz Wert"] = standardized["Wert insgesamt"].combine_first(
    standardized["Wert alternativ"]
)

# 4. Create new rows using the original table structure
new_rows = standardized[keys + ["Bilanz Wert"]].copy()
new_rows["Bilanz Position"] = position_new

# Copy metadata from an existing row for each product and year
metadata = df.drop_duplicates(keys).set_index(keys)

new_rows = new_rows.join(
    metadata[["Bilanz Oberbegriff", "Bilanz Einheit"]],
    on=keys
)

# 5. Match the original column order
new_rows = new_rows[df.columns]

# 6. Append the new rows to the original DataFrame
df = pd.concat([df, new_rows], ignore_index=True)

# 7. Check the result
print(
    df[df["Bilanz Position"] == position_new][
        ["Wirtschaftsjahr", "Bilanz Produkt", "Bilanz Position", "Bilanz Einheit", "Bilanz Wert"]
    ].head(10)
)

print("Number of new rows:", len(new_rows))

     Wirtschaftsjahr Bilanz Produkt                   Bilanz Position  \
1632         1991/92         Gerste  Inlandsverwendung standardisiert   
1633         1991/92     Kartoffeln  Inlandsverwendung standardisiert   
1634         1991/92    Weichweizen  Inlandsverwendung standardisiert   
1635         1992/93         Gerste  Inlandsverwendung standardisiert   
1636         1992/93     Kartoffeln  Inlandsverwendung standardisiert   
1637         1992/93    Weichweizen  Inlandsverwendung standardisiert   
1638         1993/94         Gerste  Inlandsverwendung standardisiert   
1639         1993/94     Kartoffeln  Inlandsverwendung standardisiert   
1640         1993/94    Weichweizen  Inlandsverwendung standardisiert   
1641         1994/95         Gerste  Inlandsverwendung standardisiert   

              Bilanz Einheit  Bilanz Wert  
1632  1 000 t Produktgewicht      10637.0  
1633  1 000 t Produktgewicht      10787.0  
1634  1 000 t Produktgewicht      11639.0  
1635  1 000 t Produk

In [6]:
# Count new rows
standardized = df[
    df["Bilanz Position"] == "Inlandsverwendung standardisiert"
]

print("Number of rows:", len(standardized))
print("Products:", standardized["Bilanz Produkt"].nunique())
print("Years:", standardized["Wirtschaftsjahr"].nunique())
print("Missing values:", standardized["Bilanz Wert"].isna().sum())

display(standardized.head(10))

Number of rows: 102
Products: 3
Years: 34
Missing values: 0


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
1632,1991/92,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10637.0
1633,1991/92,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10787.0
1634,1991/92,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11639.0
1635,1992/93,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10502.0
1636,1992/93,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,10644.0
1637,1992/93,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11004.0
1638,1993/94,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,9692.0
1639,1993/94,Kartoffeln,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,11912.0
1640,1993/94,Weichweizen,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,12943.0
1641,1994/95,Gerste,Detailposition,Inlandsverwendung standardisiert,1 000 t Produktgewicht,9901.0


In [7]:
print(df["Bilanz Produkt"].unique())

<StringArray>
['Gerste', 'Kartoffeln', 'Weichweizen']
Length: 3, dtype: str


In [8]:
# Filter the three relevant domestic use positions
# This allows us to compare the original values with the standardized values
domestic_use = df[
    df["Bilanz Position"].isin([
        "Inlandsverwendung insgesamt",
        "Inlandsverwendung",
        "Inlandsverwendung standardisiert"
    ])
]

# Summarize the available values for each balance position
# Anzahl_Werte: Number of non-missing values
# Fehlende_Werte: Number of missing values
# Anzahl_Zeilen: Total number of rows per balance position
domestic_summary = (
    domestic_use
    .groupby("Bilanz Position")
    .agg(
        Anzahl_Werte=("Bilanz Wert", "count"),
        Fehlende_Werte=("Bilanz Wert", lambda x: x.isna().sum()),
        Anzahl_Zeilen=("Bilanz Wert", "size")
    )
)

# Display the summary table
# This helps verify whether the source positions exist
# and whether the standardized values were created
display(domestic_summary)

,Anzahl_Werte,Fehlende_Werte,Anzahl_Zeilen
Bilanz Position,,,
Inlandsverwendung,34,0,34
Inlandsverwendung insgesamt,68,0,68
Inlandsverwendung standardisiert,102,0,102


In [11]:
# Define the output path
output_path = r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx"

# Save the updated DataFrame as an Excel file
df.to_excel(output_path, index=False)

# Confirm the file location
print(f"File saved: {output_path}")

File saved: C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx


In [3]:

import pandas as pd
from pathlib import Path

# Dateipfad
input_file = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx"
)

# Excel-Datei importieren
df_standardized = pd.read_excel(input_file)

# Import überprüfen
print("Datei erfolgreich importiert:")
print(input_file)

print("\nDimensionen:", df_standardized.shape)

print("\nSpaltennamen:")
print(df_standardized.columns.tolist())

# Erste Zeilen anzeigen
display(df_standardized.head())


Datei erfolgreich importiert:
C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx

Dimensionen: (1734, 6)

Spaltennamen:
['Wirtschaftsjahr', 'Bilanz Produkt', 'Bilanz Oberbegriff', 'Bilanz Position', 'Bilanz Einheit', 'Bilanz Wert']


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.0
1,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert,1 000 t Getreidewert,10.0
2,1991/92,Gerste,Detailposition,Industrielle Verwertung für Energie,1 000 t Produktgewicht,NaN
3,1991/92,Gerste,Detailposition,Saatgut,1 000 t Produktgewicht,388.0
4,1991/92,Gerste,Detailposition,Industrielle Verwertung,1 000 t Produktgewicht,2671.0


In [7]:
# Display all column names currently in memory
for col in supply_base.columns:
    print(repr(col))

'Wirtschaftsjahr'
'Bilanz_Produkt'
'Bilanz_Oberbegriff'
'Bilanz_Position'
'Bilanz_Einheit'
'Bilanz_Wert'


In [8]:

# Inspect the available balance categories and positions
print("Balance headings:")
display(supply_base["Bilanz_Oberbegriff"].value_counts(dropna=False))

print("\nBalance positions:")
display(supply_base["Bilanz_Position"].value_counts(dropna=False))

print("\nUnits:")
display(supply_base["Bilanz_Einheit"].value_counts(dropna=False))

# Search for domestic use related entries
mask = (
    supply_base["Bilanz_Oberbegriff"]
    .astype(str)
    .str.contains("Inlandsverwendung", case=False, na=False)
    |
    supply_base["Bilanz_Position"]
    .astype(str)
    .str.contains("Inlandsverwendung", case=False, na=False)
)

display(supply_base.loc[mask].head(20))


Balance headings:


Bilanz_Oberbegriff
Detailposition           918
Selbstversorgungsgrad    102
Einfuhr                  102
Verbrauch                102
Ausfuhr                  102
Verbrauch je Kopf        102
Anfangsbestand            68
Produktion                68
Endbestand                68
Bestandsänderung          34
Verwendbare Erzeugung     34
Gesamterzeugung           34
Name: count, dtype: int64


Balance positions:


Bilanz_Position
Saatgut                                                               102
Industrielle Verwertung                                               102
Verluste                                                              102
Futter                                                                102
Selbstversorgungsgrad in %                                            102
Einfuhr                                                               102
Ausfuhr                                                               102
Verwendbare Erzeugung                                                 102
Inlandsverwendung standardisiert                                      102
Inlandsverwendung insgesamt                                            68
Nahrungsverbrauch in Getreidewert                                      68
Industrielle Verwertung für Energie                                    68
Nahrungsverbrauch Ausbeute in %                                        68
Nahrungsverbrauch in G


Units:


Bilanz_Einheit
1 000 t Produktgewicht           1288
%                                 170
1 000 t Getreidewert              101
Kilogramm Mehlwert                 68
Kilogramm Getreidewert             35
Kilogramm Frischkartoffelwert      34
Kilogramm Produktgewicht           34
1 000 t Mehlwert                    4
Name: count, dtype: int64

,Wirtschaftsjahr,Bilanz_Produkt,Bilanz_Oberbegriff,Bilanz_Position,Bilanz_Einheit,Bilanz_Wert
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.00000
17,1992/93,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10502.00000
34,1993/94,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,9692.00000
51,1994/95,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,9901.00000
68,1995/96,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10523.00000
85,1996/97,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,9971.00000
102,1997/98,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10089.00000
119,1998/99,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10163.00000
136,1999/00,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,9814.00000
153,2000/01,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10004.00000


In [9]:

# Check for duplicate product-year combinations
duplicates = supply_base.duplicated(
    subset=["Wirtschaftsjahr", "Bilanz_Produkt"]
).sum()

print("Duplicate product-year combinations:", duplicates)

# Count the number of unique years per product
print("\nNumber of years per product:")
print(
    supply_base.groupby("Bilanz_Produkt")["Wirtschaftsjahr"]
    .nunique()
)

# Filter standardized domestic use records
domestic_use = supply_base.loc[
    supply_base["Bilanz_Position"].eq(
        "Inlandsverwendung standardisiert"
    )
].copy()

# Check for duplicate product-year combinations within the filtered data
domestic_duplicates = domestic_use.duplicated(
    subset=["Wirtschaftsjahr", "Bilanz_Produkt"]
).sum()

print(
    "\nDuplicate product-year combinations in standardized domestic use:",
    domestic_duplicates
)

# Create a pivot table of standardized domestic use
domestic_use_pivot = domestic_use.pivot(
    index="Wirtschaftsjahr",
    columns="Bilanz_Produkt",
    values="Bilanz_Wert"
)

display(domestic_use_pivot)


Duplicate product-year combinations: 1632

Number of years per product:
Bilanz_Produkt
Gerste         34
Kartoffeln     34
Weichweizen    34
Name: Wirtschaftsjahr, dtype: int64

Duplicate product-year combinations in standardized domestic use: 0


Bilanz_Produkt,Gerste,Kartoffeln,Weichweizen
Wirtschaftsjahr,,,
1991/92,10637.00000,10787.00000,11639.00000
1992/93,10502.00000,10644.00000,11004.00000
1993/94,9692.00000,11912.00000,12943.00000
1994/95,9901.00000,10218.00000,14470.00000
1995/96,10523.00000,10458.00000,14705.00000
1996/97,9971.00000,12223.00000,14101.00000
1997/98,10089.00000,11226.00000,14297.00000
1998/99,10163.00000,10670.00000,15285.00000
1999/00,9814.00000,10690.00000,13593.00000


In [13]:


import pandas as pd
from pathlib import Path

# Define the path to the saved Excel file
output_path = Path(
    r"C:\Users\incog\Desktop\Portfolio_Project_Agriculture\data\processed\ble_cleaned_supply_balance_standardized_domestic_use_1991-2025.xlsx"
)

# Check whether the file exists
print("File exists:", output_path.exists())

# Reload the saved Excel file without modifying it
check_df = pd.read_excel(output_path)

# Display the actual column names
print("\nSaved column names:")
print(check_df.columns.tolist())

# Display the first rows
display(check_df.head(9))


File exists: True

Saved column names:
['Wirtschaftsjahr', 'Bilanz Produkt', 'Bilanz Oberbegriff', 'Bilanz Position', 'Bilanz Einheit', 'Bilanz Wert']


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Oberbegriff,Bilanz Position,Bilanz Einheit,Bilanz Wert
0,1991/92,Gerste,Detailposition,Inlandsverwendung insgesamt,1 000 t Produktgewicht,10637.0
1,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert,1 000 t Getreidewert,10.0
2,1991/92,Gerste,Detailposition,Industrielle Verwertung für Energie,1 000 t Produktgewicht,NaN
3,1991/92,Gerste,Detailposition,Saatgut,1 000 t Produktgewicht,388.0
4,1991/92,Gerste,Detailposition,Industrielle Verwertung,1 000 t Produktgewicht,2671.0
5,1991/92,Gerste,Detailposition,Verluste,1 000 t Produktgewicht,351.0
6,1991/92,Gerste,Detailposition,Nahrungsverbrauch Ausbeute in %,%,80.0
7,1991/92,Gerste,Detailposition,Futter,1 000 t Produktgewicht,7217.0
8,1991/92,Gerste,Detailposition,Nahrungsverbrauch in Getreidewert in kg je Kopf,1 000 t Getreidewert,NaN


In [14]:

# Filter standardized domestic use records
standardized_use = check_df.loc[
    check_df["Bilanz Position"].eq(
        "Inlandsverwendung standardisiert"
    )
].copy()

# Check whether records were found
print("Number of records:", len(standardized_use))

# Check missing values
print(
    "Missing values in standardized domestic use:",
    standardized_use["Bilanz Wert"].isna().sum()
)

# Check duplicate product-year combinations
duplicates = standardized_use.duplicated(
    subset=["Wirtschaftsjahr", "Bilanz Produkt"]
).sum()

print("Duplicate product-year combinations:", duplicates)

# Display sample rows
display(
    standardized_use[
        [
            "Wirtschaftsjahr",
            "Bilanz Produkt",
            "Bilanz Einheit",
            "Bilanz Wert"
        ]
    ].head(9)
)


Number of records: 102
Missing values in standardized domestic use: 0
Duplicate product-year combinations: 0


,Wirtschaftsjahr,Bilanz Produkt,Bilanz Einheit,Bilanz Wert
1632,1991/92,Gerste,1 000 t Produktgewicht,10637.0
1633,1991/92,Kartoffeln,1 000 t Produktgewicht,10787.0
1634,1991/92,Weichweizen,1 000 t Produktgewicht,11639.0
1635,1992/93,Gerste,1 000 t Produktgewicht,10502.0
1636,1992/93,Kartoffeln,1 000 t Produktgewicht,10644.0
1637,1992/93,Weichweizen,1 000 t Produktgewicht,11004.0
1638,1993/94,Gerste,1 000 t Produktgewicht,9692.0
1639,1993/94,Kartoffeln,1 000 t Produktgewicht,11912.0
1640,1993/94,Weichweizen,1 000 t Produktgewicht,12943.0
